In [5]:
# Last update on 30/6/2026

# I've downloaded UCSC knownCanonical sequences and foldUtr3 using UCSC Table Browser on 14/1/2025.
# This version is GENCODE v47 (previous 2024 was v46).

# I'll remove "variant chromosome" (eg, chr22_KI270734v1_random).
# I'll keep only one mRNA per gene symbol and mRNA name. I'll choose ones containing the longest 3' UTR.
# 3' UTR information will be taken from foldUtr3.
# The number of non-redundant human mRNAs using this pipeline is 19,491.

In [6]:
import pandas as pd
import numpy as np

sCanonicalFile = '/home/kwonschul/data/genome/salmon_hg38_canonical_250114/Canonical_hg38_250114_v6.csv'

df = pd.read_csv(sCanonicalFile)
df

,"#""hg38.knownGene.name""",hg38.knownGene.chrom,hg38.knownGene.strand,hg38.knownGene.txStart,hg38.knownGene.txEnd,hg38.foldUtr3.name,hg38.foldUtr3.seq,hg38.kgXref.geneSymbol,hg38.kgXref.refseq,hg38.knownCanonical.chrom,hg38.knownCanonical.chromStart,hg38.knownCanonical.chromEnd,hg38.knownCanonical.transcript,hg38.knownCanonical.protein,hg38.knownGeneMrna.name,hg38.knownGeneMrna.seq
0,ENST00000832824.1,chr1,+,11120,14413,NaN,NaN,DDX11L16,NaN,NaN,NaN,NaN,NaN,NaN,ENST00000832824.1,CGCAGAGACGGGTAGAACCTCAGTAATCCGAAAAGCCGGGATCGAC...
1,ENST00000832825.1,chr1,+,11124,14405,NaN,NaN,DDX11L16,NaN,NaN,NaN,NaN,NaN,NaN,ENST00000832825.1,GAGACGGGTAGAACCTCAGTAATCCGAAAAGCCGGGATCGACCGCC...
2,ENST00000832826.1,chr1,+,11409,14413,NaN,NaN,DDX11L16,NaN,NaN,NaN,NaN,NaN,NaN,ENST00000832826.1,CCTGCTGGCAGCTGGGGACACTGCCGGGCCCTCTTGCTCCAACAGT...
3,ENST00000832827.1,chr1,+,11410,14413,NaN,NaN,DDX11L16,NaN,NaN,NaN,NaN,NaN,NaN,ENST00000832827.1,CTGCTGGCAGCTGGGGACACTGCCGGGCCCTCTTGCTCCAACAGTA...
4,ENST00000832828.1,chr1,+,11425,14409,NaN,NaN,DDX11L16,NaN,chr1,11425.0,14409.0,ENST00000832828.1,ENSG00000290825.2,ENST00000832828.1,GACACTGCCGGGCCCTCTTGCTCCAACAGTACTGGCGGATTATAGG...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
412029,ENST00000776511.1,chr22_KI270734v1_random,-,90958,98156,NaN,NaN,ENSG00000301131,NaN,chr22_KI270734v1_random,90958.0,98156.0,ENST00000776511.1,ENSG00000301131.1,ENST00000776511.1,ATTATGACGCTGTATACGACATCGCTAATGACACCGTACGAGGCAC...
412030,ENST00000839746.1,chr22_KI270734v1_random,+,109599,110784,NaN,NaN,ENSG00000309233,NaN,chr22_KI270734v1_random,109599.0,110784.0,ENST00000839746.1,ENSG00000309233.1,ENST00000839746.1,GATCCTTTTCTTTACAGGCTTCCAAGCAGAGCCCACTGAATCAAAG...
412031,ENST00000613204.1,chr22_KI270734v1_random,+,131493,137392,ENST00000613204.1,ccacaggcagcagaagucugggcagaguucaucuucuugaccuuug...,ENSG00000278817,NaN,chr22_KI270734v1_random,131493.0,137392.0,ENST00000613204.1,ENSG00000278817.1,ENST00000613204.1,CGCGAGGCGCGCCGCGATCGGGGACTGTCCTAAGACGGGCGGGGCG...
412032,ENST00000615165.1,chr22_KI270734v1_random,-,138081,161750,ENST00000615165.1,cacccgccagcacacccucagccuccagcaccccccgcccccgccc...,ENSG00000277196,NaN,NaN,NaN,NaN,NaN,NaN,ENST00000615165.1,GCTGAAGTGCCAGGGTGCTTGTGAAGTCTAAGCCCTTGTCTGGCAT...


In [7]:
# Drop pseudogenes and ncRNAs

df = df.dropna(subset=['hg38.knownCanonical.protein', 'hg38.foldUtr3.seq'])
df

,"#""hg38.knownGene.name""",hg38.knownGene.chrom,hg38.knownGene.strand,hg38.knownGene.txStart,hg38.knownGene.txEnd,hg38.foldUtr3.name,hg38.foldUtr3.seq,hg38.kgXref.geneSymbol,hg38.kgXref.refseq,hg38.knownCanonical.chrom,hg38.knownCanonical.chromStart,hg38.knownCanonical.chromEnd,hg38.knownCanonical.transcript,hg38.knownCanonical.protein,hg38.knownGeneMrna.name,hg38.knownGeneMrna.seq
333,ENST00000641515.2,chr1,+,65418,71585,ENST00000641515.2,aucuuauauaacugugagauuaaucucagauaaugacacaaaauau...,OR4F5,NM_001005484,chr1,65418.0,71585.0,ENST00000641515.2,ENSG00000186092.7,ENST00000641515.2,CCCAGATCTCTTCAGTTTTTATGCCTCATTCTGTGAAAATTGCTGT...
1201,ENST00000616016.5,chr1,+,923922,944574,ENST00000616016.5,gguugccggggguagggguggggccacacaaaucuccaggagccac...,SAMD11,NM_001385641,chr1,923922.0,944574.0,ENST00000616016.5,ENSG00000187634.13,ENST00000616016.5,GGCGGCGGAGTCTCCCAAGTCCCCGCCGGGCGGGCGCGCGCCAGTG...
1216,ENST00000327044.7,chr1,-,944202,959256,ENST00000327044.7,ggcagcccaucuggggggccuguaggggcugccgggcugguggcca...,NOC2L,NM_015658,chr1,944202.0,959256.0,ENST00000327044.7,ENSG00000188976.11,ENST00000327044.7,GCTTCGGGTTGGTGTCATGGCAGCTGCGGGGAGCCGCAAGAGGCGC...
1222,ENST00000338591.8,chr1,+,960583,965719,ENST00000338591.8,cccaccuaccaccagaggccugcagccucccacaugccuuaagggg...,KLHL17,NM_198317,chr1,960583.0,965719.0,ENST00000338591.8,ENSG00000187961.15,ENST00000338591.8,GGGAGTGAGCGACACAGAGCGGGCCGCCACCGCCGAGCAGCCCTCC...
1226,ENST00000379410.8,chr1,+,966481,975865,ENST00000379410.8,uggccgcggugagguggguucucaggaccacccucgccaagcucca...,PLEKHN1,NM_032129,chr1,966481.0,975865.0,ENST00000379410.8,ENSG00000187583.11,ENST00000379410.8,AGGAGGCTGTGGACAGGGACCCAGACTTGCCGACCTGTACGACTCT...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
411674,ENST00000613216.4,chr16_KI270728v1_random,-,1139576,1147868,ENST00000613216.4,cagcucagcucacaucaccagcucaccucugguagccauagcagcc...,ENSG00000273554,NaN,chr16_KI270728v1_random,1139576.0,1147868.0,ENST00000613216.4,ENSG00000273554.4,ENST00000613216.4,GCCGGGCCCCGCCGACGCCCGCGCGCCCCCGGGCCCCTGACACACA...
411963,ENST00000619792.1,chr22_KI270731v1_random,-,10597,13001,ENST00000619792.1,aaugccaucucgagccccuaccacucuacgggaggccucccggaac...,ENSG00000278633,NaN,chr22_KI270731v1_random,10597.0,13001.0,ENST00000619792.1,ENSG00000278633.1,ENST00000619792.1,GGCGGCTGGACGAGGACGCTCCGAGCCCAGCTCTCGAGAGTTCAAG...
412025,ENST00000617983.1,chr22_KI270734v1_random,+,72410,74814,ENST00000617983.1,aaugccaucucaagccccuaccgcucuacgggaggccucccggaac...,ENSG00000276017,NaN,chr22_KI270734v1_random,72410.0,74814.0,ENST00000617983.1,ENSG00000276017.1,ENST00000617983.1,GGCGGCTGGACGAGGACGCTCCGAGCCCAGCTCTCGAGAGTTCAAG...
412031,ENST00000613204.1,chr22_KI270734v1_random,+,131493,137392,ENST00000613204.1,ccacaggcagcagaagucugggcagaguucaucuucuugaccuuug...,ENSG00000278817,NaN,chr22_KI270734v1_random,131493.0,137392.0,ENST00000613204.1,ENSG00000278817.1,ENST00000613204.1,CGCGAGGCGCGCCGCGATCGGGGACTGTCCTAAGACGGGCGGGGCG...


In [8]:
df = df.loc[(df['hg38.knownGene.chrom'].str.len() <= 5)] 
df

,"#""hg38.knownGene.name""",hg38.knownGene.chrom,hg38.knownGene.strand,hg38.knownGene.txStart,hg38.knownGene.txEnd,hg38.foldUtr3.name,hg38.foldUtr3.seq,hg38.kgXref.geneSymbol,hg38.kgXref.refseq,hg38.knownCanonical.chrom,hg38.knownCanonical.chromStart,hg38.knownCanonical.chromEnd,hg38.knownCanonical.transcript,hg38.knownCanonical.protein,hg38.knownGeneMrna.name,hg38.knownGeneMrna.seq
333,ENST00000641515.2,chr1,+,65418,71585,ENST00000641515.2,aucuuauauaacugugagauuaaucucagauaaugacacaaaauau...,OR4F5,NM_001005484,chr1,65418.0,71585.0,ENST00000641515.2,ENSG00000186092.7,ENST00000641515.2,CCCAGATCTCTTCAGTTTTTATGCCTCATTCTGTGAAAATTGCTGT...
1201,ENST00000616016.5,chr1,+,923922,944574,ENST00000616016.5,gguugccggggguagggguggggccacacaaaucuccaggagccac...,SAMD11,NM_001385641,chr1,923922.0,944574.0,ENST00000616016.5,ENSG00000187634.13,ENST00000616016.5,GGCGGCGGAGTCTCCCAAGTCCCCGCCGGGCGGGCGCGCGCCAGTG...
1216,ENST00000327044.7,chr1,-,944202,959256,ENST00000327044.7,ggcagcccaucuggggggccuguaggggcugccgggcugguggcca...,NOC2L,NM_015658,chr1,944202.0,959256.0,ENST00000327044.7,ENSG00000188976.11,ENST00000327044.7,GCTTCGGGTTGGTGTCATGGCAGCTGCGGGGAGCCGCAAGAGGCGC...
1222,ENST00000338591.8,chr1,+,960583,965719,ENST00000338591.8,cccaccuaccaccagaggccugcagccucccacaugccuuaagggg...,KLHL17,NM_198317,chr1,960583.0,965719.0,ENST00000338591.8,ENSG00000187961.15,ENST00000338591.8,GGGAGTGAGCGACACAGAGCGGGCCGCCACCGCCGAGCAGCCCTCC...
1226,ENST00000379410.8,chr1,+,966481,975865,ENST00000379410.8,uggccgcggugagguggguucucaggaccacccucgccaagcucca...,PLEKHN1,NM_032129,chr1,966481.0,975865.0,ENST00000379410.8,ENSG00000187583.11,ENST00000379410.8,AGGAGGCTGTGGACAGGGACCCAGACTTGCCGACCTGTACGACTCT...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
385484,ENST00000329492.6,chr22,+,50600792,50613978,ENST00000329492.6,ccugcccaccgcucugucuccuggccgucugcuccaggcgcagcug...,MAPK8IP2,NM_012324,chr22,50600792.0,50613978.0,ENST00000329492.6,ENSG00000008735.14,ENST00000329492.6,GCAGTCGCGGGCCTCTCCCGGAGAAGATGGCGGATCGCGCGGAGAT...
385488,ENST00000216124.10,chr22,-,50622753,50628152,ENST00000216124.10,gggccccucggcuggccugggcaugugauggcuccucacugggagc...,ARSA,NM_001085427,chr22,50622753.0,50628152.0,ENST00000216124.10,ENSG00000100299.18,ENST00000216124.10,GAGGCGGAAGCGCCCGCAGCCCGGTACCGGCTCCTCCTGGGCTCCC...
385516,ENST00000692848.2,chr22,+,50674407,50733212,ENST00000692848.2,cgccccacccccacucccgccccggccgugcccugccggcagggcc...,SHANK3,NM_001372044,chr22,50674407.0,50733212.0,ENST00000692848.2,ENSG00000251322.12,ENST00000692848.2,GGGGCCGATGCAGCTGAGCCGCGCCGCCGCCGCCGCCGCCGCCGCC...
385528,ENST00000216139.10,chr22,+,50738203,50745339,ENST00000216139.10,ucugaccugguucucaacagacccagugagcccuucacuccugaga...,ACR,NM_001097,chr22,50738203.0,50745339.0,ENST00000216139.10,ENSG00000100312.11,ENST00000216139.10,ACTAGGCTTGCAGGCCAGGCAGTGCCAGGAGTATGGTTGAGATGCT...


In [9]:
df['id'] = df.apply(lambda row: row['hg38.kgXref.geneSymbol'] + '|' + \
                                row['hg38.knownCanonical.protein'] + '|' + \
                                row['hg38.knownCanonical.transcript'], axis=1)
df

/tmp/ipykernel_3712785/247583392.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['id'] = df.apply(lambda row: row['hg38.kgXref.geneSymbol'] + '|' + \


,"#""hg38.knownGene.name""",hg38.knownGene.chrom,hg38.knownGene.strand,hg38.knownGene.txStart,hg38.knownGene.txEnd,hg38.foldUtr3.name,hg38.foldUtr3.seq,hg38.kgXref.geneSymbol,hg38.kgXref.refseq,hg38.knownCanonical.chrom,hg38.knownCanonical.chromStart,hg38.knownCanonical.chromEnd,hg38.knownCanonical.transcript,hg38.knownCanonical.protein,hg38.knownGeneMrna.name,hg38.knownGeneMrna.seq,id
333,ENST00000641515.2,chr1,+,65418,71585,ENST00000641515.2,aucuuauauaacugugagauuaaucucagauaaugacacaaaauau...,OR4F5,NM_001005484,chr1,65418.0,71585.0,ENST00000641515.2,ENSG00000186092.7,ENST00000641515.2,CCCAGATCTCTTCAGTTTTTATGCCTCATTCTGTGAAAATTGCTGT...,OR4F5|ENSG00000186092.7|ENST00000641515.2
1201,ENST00000616016.5,chr1,+,923922,944574,ENST00000616016.5,gguugccggggguagggguggggccacacaaaucuccaggagccac...,SAMD11,NM_001385641,chr1,923922.0,944574.0,ENST00000616016.5,ENSG00000187634.13,ENST00000616016.5,GGCGGCGGAGTCTCCCAAGTCCCCGCCGGGCGGGCGCGCGCCAGTG...,SAMD11|ENSG00000187634.13|ENST00000616016.5
1216,ENST00000327044.7,chr1,-,944202,959256,ENST00000327044.7,ggcagcccaucuggggggccuguaggggcugccgggcugguggcca...,NOC2L,NM_015658,chr1,944202.0,959256.0,ENST00000327044.7,ENSG00000188976.11,ENST00000327044.7,GCTTCGGGTTGGTGTCATGGCAGCTGCGGGGAGCCGCAAGAGGCGC...,NOC2L|ENSG00000188976.11|ENST00000327044.7
1222,ENST00000338591.8,chr1,+,960583,965719,ENST00000338591.8,cccaccuaccaccagaggccugcagccucccacaugccuuaagggg...,KLHL17,NM_198317,chr1,960583.0,965719.0,ENST00000338591.8,ENSG00000187961.15,ENST00000338591.8,GGGAGTGAGCGACACAGAGCGGGCCGCCACCGCCGAGCAGCCCTCC...,KLHL17|ENSG00000187961.15|ENST00000338591.8
1226,ENST00000379410.8,chr1,+,966481,975865,ENST00000379410.8,uggccgcggugagguggguucucaggaccacccucgccaagcucca...,PLEKHN1,NM_032129,chr1,966481.0,975865.0,ENST00000379410.8,ENSG00000187583.11,ENST00000379410.8,AGGAGGCTGTGGACAGGGACCCAGACTTGCCGACCTGTACGACTCT...,PLEKHN1|ENSG00000187583.11|ENST00000379410.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
385484,ENST00000329492.6,chr22,+,50600792,50613978,ENST00000329492.6,ccugcccaccgcucugucuccuggccgucugcuccaggcgcagcug...,MAPK8IP2,NM_012324,chr22,50600792.0,50613978.0,ENST00000329492.6,ENSG00000008735.14,ENST00000329492.6,GCAGTCGCGGGCCTCTCCCGGAGAAGATGGCGGATCGCGCGGAGAT...,MAPK8IP2|ENSG00000008735.14|ENST00000329492.6
385488,ENST00000216124.10,chr22,-,50622753,50628152,ENST00000216124.10,gggccccucggcuggccugggcaugugauggcuccucacugggagc...,ARSA,NM_001085427,chr22,50622753.0,50628152.0,ENST00000216124.10,ENSG00000100299.18,ENST00000216124.10,GAGGCGGAAGCGCCCGCAGCCCGGTACCGGCTCCTCCTGGGCTCCC...,ARSA|ENSG00000100299.18|ENST00000216124.10
385516,ENST00000692848.2,chr22,+,50674407,50733212,ENST00000692848.2,cgccccacccccacucccgccccggccgugcccugccggcagggcc...,SHANK3,NM_001372044,chr22,50674407.0,50733212.0,ENST00000692848.2,ENSG00000251322.12,ENST00000692848.2,GGGGCCGATGCAGCTGAGCCGCGCCGCCGCCGCCGCCGCCGCCGCC...,SHANK3|ENSG00000251322.12|ENST00000692848.2
385528,ENST00000216139.10,chr22,+,50738203,50745339,ENST00000216139.10,ucugaccugguucucaacagacccagugagcccuucacuccugaga...,ACR,NM_001097,chr22,50738203.0,50745339.0,ENST00000216139.10,ENSG00000100312.11,ENST00000216139.10,ACTAGGCTTGCAGGCCAGGCAGTGCCAGGAGTATGGTTGAGATGCT...,ACR|ENSG00000100312.11|ENST00000216139.10


In [10]:
# Remove duplicated items

In [11]:
len(df['hg38.kgXref.geneSymbol'])-len(df['hg38.kgXref.geneSymbol'].drop_duplicates())

24

In [12]:
df[df.duplicated(['hg38.kgXref.geneSymbol'], keep=False)]

,"#""hg38.knownGene.name""",hg38.knownGene.chrom,hg38.knownGene.strand,hg38.knownGene.txStart,hg38.knownGene.txEnd,hg38.foldUtr3.name,hg38.foldUtr3.seq,hg38.kgXref.geneSymbol,hg38.kgXref.refseq,hg38.knownCanonical.chrom,hg38.knownCanonical.chromStart,hg38.knownCanonical.chromEnd,hg38.knownCanonical.transcript,hg38.knownCanonical.protein,hg38.knownGeneMrna.name,hg38.knownGeneMrna.seq,id
95391,ENST00000295898.8,chr4,-,86876204,86892422,ENST00000295898.8,caccugaagcgcggauuguagaugaagaaacucagcagcgccggcg...,C4orf36,NM_144645,chr4,86876204.0,86892422.0,ENST00000295898.8,ENSG00000163633.12,ENST00000295898.8,AGACTCGGCGCCCGCGGCGAGCGGGAAGGAGGCGTGTGCCTGCGGC...,C4orf36|ENSG00000163633.12|ENST00000295898.8
95393,ENST00000473559.5,chr4,-,86886471,86934732,ENST00000473559.5,cagucaguucaugauugaaucuguacucugugucuugcauccaaag...,C4orf36,NR_183022,chr4,86886471.0,86934732.0,ENST00000473559.5,ENSG00000285458.1,ENST00000473559.5,ACTTTACGGCGAGATTAAGCTCAGGGTCAAGTTCTGAGCAAAAAGT...,C4orf36|ENSG00000285458.1|ENST00000473559.5
95706,ENST00000512194.2,chr4,+,88523809,88708450,ENST00000512194.2,ggcuucucagcuuguccaguauuucccuucguuccucaguguccac...,HERC3,NM_001375483,chr4,88523809.0,88708450.0,ENST00000512194.2,ENSG00000287542.1,ENST00000512194.2,AGGAAACGCTGCCGAGGAGTAGGCGGGGCGGGCGCTTTGGTTCCGA...,HERC3|ENSG00000287542.1|ENST00000512194.2
95712,ENST00000402738.6,chr4,+,88592479,88708539,ENST00000402738.6,ggcuucucagcuuguccaguauuucccuucguuccucaguguccac...,HERC3,NM_014606,chr4,88592479.0,88708539.0,ENST00000402738.6,ENSG00000138641.19,ENST00000402738.6,CTTATTATTGTGACGCCGAAAACGGAGAAACCCCGGGTCCGGCGAG...,HERC3|ENSG00000138641.19|ENST00000402738.6
112407,ENST00000646262.1,chr5,+,77086731,77427124,ENST00000646262.1,agccaagccacagagggggccucuugaccgacaaaggacacuguga...,PDE8B,NM_001414623,chr5,77086731.0,77427124.0,ENST00000646262.1,ENSG00000284762.1,ENST00000646262.1,AGTTACAGAAAGAACCCACTGCAGGGCGTCCAGTCACTTGCCCAGT...,PDE8B|ENSG00000284762.1|ENST00000646262.1
112450,ENST00000264917.10,chr5,+,77210679,77428256,ENST00000264917.10,agccaagccacagagggggccucuugaccgacaaaggacacuguga...,PDE8B,NM_003719,chr5,77210679.0,77428256.0,ENST00000264917.10,ENSG00000113231.14,ENST00000264917.10,GTGTATGCGCGCTCCCCCGCTCGGGGAGGAAGATGGCCCAAAAGGG...,PDE8B|ENSG00000113231.14|ENST00000264917.10
118216,ENST00000361059.7,chr5,+,139274102,139330394,ENST00000361059.7,gaugugcaaggagauuuaaugauuucaaagaaaauaaugguucuuu...,MATR3,NM_199189,chr5,139274102.0,139330394.0,ENST00000361059.7,ENSG00000280987.4,ENST00000361059.7,CTTTCCCCTCCGGCCTCTGCCGGTGCTGCTGCGCCCTGCGGAGCTC...,MATR3|ENSG00000280987.4|ENST00000361059.7
118239,ENST00000394805.8,chr5,+,139293739,139331677,ENST00000394805.8,gaugugcaaggagauuuaaugauuucaaagaaaauaaugguucuuu...,MATR3,NM_018834,chr5,139293739.0,139331677.0,ENST00000394805.8,ENSG00000015479.20,ENST00000394805.8,GGGAGTCTCCGCGTCCCGCTCGCTGGGAGAGAGGTACCTCTCCTTT...,MATR3|ENSG00000015479.20|ENST00000394805.8
166649,ENST00000554914.1,chr8,-,10725398,10839847,ENST00000554914.1,agcuggaggcgccccguccggucagcccucgcgcccucuccuucuu...,PINX1,NaN,chr8,10725398.0,10839847.0,ENST00000554914.1,ENSG00000258724.1,ENST00000554914.1,TACGCACGTCCTGATTCTCCTGGAGTCTCCAGCCCGCCCAGTGGCC...,PINX1|ENSG00000258724.1|ENST00000554914.1
166650,ENST00000314787.8,chr8,-,10764960,10839875,ENST00000314787.8,auccuucccagccggggccuuccgaccacucagcugucagggcacu...,PINX1,NM_017884,chr8,10764960.0,10839875.0,ENST00000314787.8,ENSG00000254093.9,ENST00000314787.8,GTCCGCTGCAGTCCGCCGGCGAGGGAGTTACGCACGTCCTGATTCT...,PINX1|ENSG00000254093.9|ENST00000314787.8


In [13]:
df['hg38.kgXref.refseq'].isna().sum()

502

In [14]:
# sort according to the 3' UTR length
# keep only one item for each gene symbol

In [15]:
df['3utr_len'] = df.apply(lambda row: len(row['hg38.foldUtr3.seq']), axis=1)
df

/tmp/ipykernel_3712785/4132773710.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['3utr_len'] = df.apply(lambda row: len(row['hg38.foldUtr3.seq']), axis=1)


,"#""hg38.knownGene.name""",hg38.knownGene.chrom,hg38.knownGene.strand,hg38.knownGene.txStart,hg38.knownGene.txEnd,hg38.foldUtr3.name,hg38.foldUtr3.seq,hg38.kgXref.geneSymbol,hg38.kgXref.refseq,hg38.knownCanonical.chrom,hg38.knownCanonical.chromStart,hg38.knownCanonical.chromEnd,hg38.knownCanonical.transcript,hg38.knownCanonical.protein,hg38.knownGeneMrna.name,hg38.knownGeneMrna.seq,id,3utr_len
333,ENST00000641515.2,chr1,+,65418,71585,ENST00000641515.2,aucuuauauaacugugagauuaaucucagauaaugacacaaaauau...,OR4F5,NM_001005484,chr1,65418.0,71585.0,ENST00000641515.2,ENSG00000186092.7,ENST00000641515.2,CCCAGATCTCTTCAGTTTTTATGCCTCATTCTGTGAAAATTGCTGT...,OR4F5|ENSG00000186092.7|ENST00000641515.2,1577
1201,ENST00000616016.5,chr1,+,923922,944574,ENST00000616016.5,gguugccggggguagggguggggccacacaaaucuccaggagccac...,SAMD11,NM_001385641,chr1,923922.0,944574.0,ENST00000616016.5,ENSG00000187634.13,ENST00000616016.5,GGCGGCGGAGTCTCCCAAGTCCCCGCCGGGCGGGCGCGCGCCAGTG...,SAMD11|ENSG00000187634.13|ENST00000616016.5,421
1216,ENST00000327044.7,chr1,-,944202,959256,ENST00000327044.7,ggcagcccaucuggggggccuguaggggcugccgggcugguggcca...,NOC2L,NM_015658,chr1,944202.0,959256.0,ENST00000327044.7,ENSG00000188976.11,ENST00000327044.7,GCTTCGGGTTGGTGTCATGGCAGCTGCGGGGAGCCGCAAGAGGCGC...,NOC2L|ENSG00000188976.11|ENST00000327044.7,491
1222,ENST00000338591.8,chr1,+,960583,965719,ENST00000338591.8,cccaccuaccaccagaggccugcagccucccacaugccuuaagggg...,KLHL17,NM_198317,chr1,960583.0,965719.0,ENST00000338591.8,ENSG00000187961.15,ENST00000338591.8,GGGAGTGAGCGACACAGAGCGGGCCGCCACCGCCGAGCAGCCCTCC...,KLHL17|ENSG00000187961.15|ENST00000338591.8,528
1226,ENST00000379410.8,chr1,+,966481,975865,ENST00000379410.8,uggccgcggugagguggguucucaggaccacccucgccaagcucca...,PLEKHN1,NM_032129,chr1,966481.0,975865.0,ENST00000379410.8,ENSG00000187583.11,ENST00000379410.8,AGGAGGCTGTGGACAGGGACCCAGACTTGCCGACCTGTACGACTCT...,PLEKHN1|ENSG00000187583.11|ENST00000379410.8,1290
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
385484,ENST00000329492.6,chr22,+,50600792,50613978,ENST00000329492.6,ccugcccaccgcucugucuccuggccgucugcuccaggcgcagcug...,MAPK8IP2,NM_012324,chr22,50600792.0,50613978.0,ENST00000329492.6,ENSG00000008735.14,ENST00000329492.6,GCAGTCGCGGGCCTCTCCCGGAGAAGATGGCGGATCGCGCGGAGAT...,MAPK8IP2|ENSG00000008735.14|ENST00000329492.6,3199
385488,ENST00000216124.10,chr22,-,50622753,50628152,ENST00000216124.10,gggccccucggcuggccugggcaugugauggcuccucacugggagc...,ARSA,NM_001085427,chr22,50622753.0,50628152.0,ENST00000216124.10,ENSG00000100299.18,ENST00000216124.10,GAGGCGGAAGCGCCCGCAGCCCGGTACCGGCTCCTCCTGGGCTCCC...,ARSA|ENSG00000100299.18|ENST00000216124.10,2391
385516,ENST00000692848.2,chr22,+,50674407,50733212,ENST00000692848.2,cgccccacccccacucccgccccggccgugcccugccggcagggcc...,SHANK3,NM_001372044,chr22,50674407.0,50733212.0,ENST00000692848.2,ENSG00000251322.12,ENST00000692848.2,GGGGCCGATGCAGCTGAGCCGCGCCGCCGCCGCCGCCGCCGCCGCC...,SHANK3|ENSG00000251322.12|ENST00000692848.2,1900
385528,ENST00000216139.10,chr22,+,50738203,50745339,ENST00000216139.10,ucugaccugguucucaacagacccagugagcccuucacuccugaga...,ACR,NM_001097,chr22,50738203.0,50745339.0,ENST00000216139.10,ENSG00000100312.11,ENST00000216139.10,ACTAGGCTTGCAGGCCAGGCAGTGCCAGGAGTATGGTTGAGATGCT...,ACR|ENSG00000100312.11|ENST00000216139.10,132


In [16]:
df = df.sort_values(by=['3utr_len'])
df

,"#""hg38.knownGene.name""",hg38.knownGene.chrom,hg38.knownGene.strand,hg38.knownGene.txStart,hg38.knownGene.txEnd,hg38.foldUtr3.name,hg38.foldUtr3.seq,hg38.kgXref.geneSymbol,hg38.kgXref.refseq,hg38.knownCanonical.chrom,hg38.knownCanonical.chromStart,hg38.knownCanonical.chromEnd,hg38.knownCanonical.transcript,hg38.knownCanonical.protein,hg38.knownGeneMrna.name,hg38.knownGeneMrna.seq,id,3utr_len
116277,ENST00000509403.7,chr5,+,122160175,122182658,ENST00000509403.7,agcccuuuucucucua,ZNF475,NM_001195535,chr5,122160175.0,122182658.0,ENST00000509403.7,ENSG00000250803.7,ENST00000509403.7,ATCTTTCCTTGGTGCCAACCATGCCTGCATATCAGGGCAGCATTCT...,ZNF475|ENSG00000250803.7|ENST00000509403.7,16
350294,ENST00000263897.10,chr19,+,17215356,17219829,ENST00000263897.10,agacccccgcccaccu,USE1,NM_018467,chr19,17215356.0,17219829.0,ENST00000263897.10,ENSG00000053501.15,ENST00000263897.10,CTCTCTTAACATGGAGCCGGCGGAAGGGGTGGTGTAGGGCCGGGCG...,USE1|ENSG00000053501.15|ENST00000263897.10,16
359177,ENST00000391826.7,chr19,-,49889653,49896887,ENST00000391826.7,aguauuuucggaaaaa,IL4I1,NM_152899,chr19,49889653.0,49896887.0,ENST00000391826.7,ENSG00000104951.16,ENST00000391826.7,AGTGGAGGGCAGTGGAGAGGACCGCGCTGTCCTGCTGTCACCAAGA...,IL4I1|ENSG00000104951.16|ENST00000391826.7,16
48238,ENST00000376865.1,chr2,+,100470481,100483280,ENST00000376865.1,aagaaagcuggaucug,NMS,NM_001011717,chr2,100470481.0,100483280.0,ENST00000376865.1,ENSG00000204640.1,ENST00000376865.1,TCTCACAATGAAACATCTTCGTCCCCAGTTCCCTCTCATCTTGGCC...,NMS|ENSG00000204640.1|ENST00000376865.1,16
319748,ENST00000396463.7,chr17,+,7688476,7703502,ENST00000396463.7,aaagguuuuuaugaua,WRAP53,NM_018081,chr17,7688476.0,7703502.0,ENST00000396463.7,ENSG00000141499.18,ENST00000396463.7,AGTGCTTTCAAAAGAATTGGCGTCCGCTGTTCGCCTCTCCTCCCGG...,WRAP53|ENSG00000141499.18|ENST00000396463.7,16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259269,ENST00000611864.5,chr12,-,85955666,86256391,ENST00000611864.5,ccaauuaaaucaguauguucaguuucugaagcaguucuuccugcuu...,MGAT4C,NR_147093,chr12,85955666.0,86256391.0,ENST00000611864.5,ENSG00000182050.14,ENST00000611864.5,AGTTTCAGCACAGTGCCATTATGATGGATGTTTACATCTCTTCATT...,MGAT4C|ENSG00000182050.14|ENST00000611864.5,23622
283870,ENST00000614987.5,chr14,-,90847860,91060641,ENST00000614987.5,gcaugguaggaguguaucagugauccauugcaccuuuauucccuca...,RPS6KA5,NM_004755,chr14,90847860.0,91060641.0,ENST00000614987.5,ENSG00000100784.12,ENST00000614987.5,GCGGTACTAGCGGTGCCCGCCGAAGGGGGAGGAGGCGAGGAGCGAG...,RPS6KA5|ENSG00000100784.12|ENST00000614987.5,24213
76230,ENST00000675478.1,chr3,-,114314499,115147288,ENST00000675478.1,guaguaucuuucucucuuucuuaugaacaaaacaaaacaacaacaa...,ZBTB20,NM_001348803,chr3,114314499.0,115147288.0,ENST00000675478.1,ENSG00000181722.18,ENST00000675478.1,CTTCTGCCGAAGAAGTAGGGGCGGGGGGAAGTTTAGGAGTTGAGGA...,ZBTB20|ENSG00000181722.18|ENST00000675478.1,24505
250270,ENST00000609686.4,chr12,-,13537336,13981602,ENST00000609686.4,gugagggaacagagagguuaagguggguacgggaggguaaggcugu...,GRIN2B,NM_000834,chr12,13537336.0,13981602.0,ENST00000609686.4,ENSG00000273079.7,ENST00000609686.4,ACTTCCTCCGGCTTCTTCACCTCTAGGAAATCTCGGGGCTTCGCTC...,GRIN2B|ENSG00000273079.7|ENST00000609686.4,25446


In [17]:
df = df.drop_duplicates('hg38.kgXref.geneSymbol', keep='last')
df

,"#""hg38.knownGene.name""",hg38.knownGene.chrom,hg38.knownGene.strand,hg38.knownGene.txStart,hg38.knownGene.txEnd,hg38.foldUtr3.name,hg38.foldUtr3.seq,hg38.kgXref.geneSymbol,hg38.kgXref.refseq,hg38.knownCanonical.chrom,hg38.knownCanonical.chromStart,hg38.knownCanonical.chromEnd,hg38.knownCanonical.transcript,hg38.knownCanonical.protein,hg38.knownGeneMrna.name,hg38.knownGeneMrna.seq,id,3utr_len
116277,ENST00000509403.7,chr5,+,122160175,122182658,ENST00000509403.7,agcccuuuucucucua,ZNF475,NM_001195535,chr5,122160175.0,122182658.0,ENST00000509403.7,ENSG00000250803.7,ENST00000509403.7,ATCTTTCCTTGGTGCCAACCATGCCTGCATATCAGGGCAGCATTCT...,ZNF475|ENSG00000250803.7|ENST00000509403.7,16
350294,ENST00000263897.10,chr19,+,17215356,17219829,ENST00000263897.10,agacccccgcccaccu,USE1,NM_018467,chr19,17215356.0,17219829.0,ENST00000263897.10,ENSG00000053501.15,ENST00000263897.10,CTCTCTTAACATGGAGCCGGCGGAAGGGGTGGTGTAGGGCCGGGCG...,USE1|ENSG00000053501.15|ENST00000263897.10,16
359177,ENST00000391826.7,chr19,-,49889653,49896887,ENST00000391826.7,aguauuuucggaaaaa,IL4I1,NM_152899,chr19,49889653.0,49896887.0,ENST00000391826.7,ENSG00000104951.16,ENST00000391826.7,AGTGGAGGGCAGTGGAGAGGACCGCGCTGTCCTGCTGTCACCAAGA...,IL4I1|ENSG00000104951.16|ENST00000391826.7,16
48238,ENST00000376865.1,chr2,+,100470481,100483280,ENST00000376865.1,aagaaagcuggaucug,NMS,NM_001011717,chr2,100470481.0,100483280.0,ENST00000376865.1,ENSG00000204640.1,ENST00000376865.1,TCTCACAATGAAACATCTTCGTCCCCAGTTCCCTCTCATCTTGGCC...,NMS|ENSG00000204640.1|ENST00000376865.1,16
319748,ENST00000396463.7,chr17,+,7688476,7703502,ENST00000396463.7,aaagguuuuuaugaua,WRAP53,NM_018081,chr17,7688476.0,7703502.0,ENST00000396463.7,ENSG00000141499.18,ENST00000396463.7,AGTGCTTTCAAAAGAATTGGCGTCCGCTGTTCGCCTCTCCTCCCGG...,WRAP53|ENSG00000141499.18|ENST00000396463.7,16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259269,ENST00000611864.5,chr12,-,85955666,86256391,ENST00000611864.5,ccaauuaaaucaguauguucaguuucugaagcaguucuuccugcuu...,MGAT4C,NR_147093,chr12,85955666.0,86256391.0,ENST00000611864.5,ENSG00000182050.14,ENST00000611864.5,AGTTTCAGCACAGTGCCATTATGATGGATGTTTACATCTCTTCATT...,MGAT4C|ENSG00000182050.14|ENST00000611864.5,23622
283870,ENST00000614987.5,chr14,-,90847860,91060641,ENST00000614987.5,gcaugguaggaguguaucagugauccauugcaccuuuauucccuca...,RPS6KA5,NM_004755,chr14,90847860.0,91060641.0,ENST00000614987.5,ENSG00000100784.12,ENST00000614987.5,GCGGTACTAGCGGTGCCCGCCGAAGGGGGAGGAGGCGAGGAGCGAG...,RPS6KA5|ENSG00000100784.12|ENST00000614987.5,24213
76230,ENST00000675478.1,chr3,-,114314499,115147288,ENST00000675478.1,guaguaucuuucucucuuucuuaugaacaaaacaaaacaacaacaa...,ZBTB20,NM_001348803,chr3,114314499.0,115147288.0,ENST00000675478.1,ENSG00000181722.18,ENST00000675478.1,CTTCTGCCGAAGAAGTAGGGGCGGGGGGAAGTTTAGGAGTTGAGGA...,ZBTB20|ENSG00000181722.18|ENST00000675478.1,24505
250270,ENST00000609686.4,chr12,-,13537336,13981602,ENST00000609686.4,gugagggaacagagagguuaagguggguacgggaggguaaggcugu...,GRIN2B,NM_000834,chr12,13537336.0,13981602.0,ENST00000609686.4,ENSG00000273079.7,ENST00000609686.4,ACTTCCTCCGGCTTCTTCACCTCTAGGAAATCTCGGGGCTTCGCTC...,GRIN2B|ENSG00000273079.7|ENST00000609686.4,25446


In [18]:
# sort according to the alphabetical order of gene symbol
# remove duplicates according to the mRNA sequence (copy number variations; eg, AMY1A, AMY1C..)



In [19]:
df = df.sort_values(by=['hg38.kgXref.geneSymbol'])
df

,"#""hg38.knownGene.name""",hg38.knownGene.chrom,hg38.knownGene.strand,hg38.knownGene.txStart,hg38.knownGene.txEnd,hg38.foldUtr3.name,hg38.foldUtr3.seq,hg38.kgXref.geneSymbol,hg38.kgXref.refseq,hg38.knownCanonical.chrom,hg38.knownCanonical.chromStart,hg38.knownCanonical.chromEnd,hg38.knownCanonical.transcript,hg38.knownCanonical.protein,hg38.knownGeneMrna.name,hg38.knownGeneMrna.seq,id,3utr_len
362233,ENST00000263100.8,chr19,-,58345182,58353492,ENST00000263100.8,ugcagccgcgggcccagggugcuguugguguccucagaagugccgg...,A1BG,NM_130786,chr19,58345182.0,58353492.0,ENST00000263100.8,ENSG00000121410.12,ENST00000263100.8,ATTGCTGCAGACGCTCACCCCAGACACTCACTGCACCGGAGTGAGC...,A1BG|ENSG00000121410.12|ENST00000263100.8,1839
217289,ENST00000373997.8,chr10,-,50799408,50885627,ENST00000373997.8,agaugcuuuuuuaaauuuaagaauaagacacacaaaacucuauuaa...,A1CF,NM_014576,chr10,50799408.0,50885627.0,ENST00000373997.8,ENSG00000148584.16,ENST00000373997.8,ATAATCAAGGAAACCTTTTCCGGGTGGGGATCTCTGAAATTACTCA...,A1CF|ENSG00000148584.16|ENST00000373997.8,7320
249300,ENST00000318602.12,chr12,-,9067707,9115919,ENST00000318602.12,agaccacaaggcugaaaagugcuuugcuggaguccuguucucagag...,A2M,NM_001347425,chr12,9067707.0,9115919.0,ENST00000318602.12,ENSG00000175899.15,ENST00000318602.12,GGGACCAGATGGATTGTAGGGAGTAGGGTACAATACAGTCTGTTCT...,A2M|ENSG00000175899.15|ENST00000318602.12,115
249241,ENST00000299698.12,chr12,+,8822620,8876787,ENST00000299698.12,ggauaggagcuggaaacucaauuaguccucugugacauuuacugga...,A2ML1,NM_144670,chr12,8822620.0,8876787.0,ENST00000299698.12,ENSG00000166535.20,ENST00000299698.12,GACCCTGGAAAAATCTGTCTCACCCACAAAGATGTGGGCTCAGCTC...,A2ML1|ENSG00000166535.20|ENST00000299698.12,731
384208,ENST00000642412.2,chr22,-,42692120,42720870,ENST00000642412.2,ggggcccgccaggucaccuccccaaccugcuccugauggggcacug...,A4GALT,NM_017436,chr22,42692120.0,42720870.0,ENST00000642412.2,ENSG00000128274.17,ENST00000642412.2,ACTGCCCTCCGGCCGCCGCGCCGCCCGCCCGCCGGGGCCCCGCTGT...,A4GALT|ENSG00000128274.17|ENST00000642412.2,769
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10748,ENST00000371528.2,chr1,+,52842759,52894998,ENST00000371528.2,accuaaggaauuucagaggugugugcucuuccucaaugucaggugu...,ZYG11A,NM_001004339,chr1,52842759.0,52894998.0,ENST00000371528.2,ENSG00000203995.11,ENST00000371528.2,GCTAGCTCCGTGTGCCTCGCAGGCGTGGTGGGCGCGTCCTGGCAGC...,ZYG11A|ENSG00000203995.11|ENST00000371528.2,2041
10742,ENST00000294353.7,chr1,+,52726452,52827336,ENST00000294353.7,uagccauaaguauuggauaguugaaucacaggaauccuuuuuguga...,ZYG11B,NM_024646,chr1,52726452.0,52827336.0,ENST00000294353.7,ENSG00000162378.13,ENST00000294353.7,GGAGTCTGCGCTCTGGTTCGGGCTGCGGCTGCGGCTGCGGCTGCGG...,ZYG11B|ENSG00000162378.13|ENST00000294353.7,5707
162759,ENST00000322764.10,chr7,+,143381344,143391111,ENST00000322764.10,gugaggacaggcccucuucagaccgcaguccaugccccauugugga...,ZYX,NM_003461,chr7,143381344.0,143391111.0,ENST00000322764.10,ENSG00000159840.16,ENST00000322764.10,GCAGAGTCTGCGGACCCGGCGCCGAGGCGGCCACCCGAGACGCGGC...,ZYX|ENSG00000159840.16|ENST00000322764.10,429
318347,ENST00000381638.7,chr17,-,4004444,4143030,ENST00000381638.7,cccuugguggggcgugcaugggacuaguucaucugccacagggauu...,ZZEF1,NM_015113,chr17,4004444.0,4143030.0,ENST00000381638.7,ENSG00000074755.15,ENST00000381638.7,AGGAAGCCGGAAGCCGCAGGGGCCGCCGTCGTCTCCTCCGCGTCCC...,ZZEF1|ENSG00000074755.15|ENST00000381638.7,2445


In [20]:
df = df.drop_duplicates('hg38.knownGeneMrna.seq', keep='first')
df

,"#""hg38.knownGene.name""",hg38.knownGene.chrom,hg38.knownGene.strand,hg38.knownGene.txStart,hg38.knownGene.txEnd,hg38.foldUtr3.name,hg38.foldUtr3.seq,hg38.kgXref.geneSymbol,hg38.kgXref.refseq,hg38.knownCanonical.chrom,hg38.knownCanonical.chromStart,hg38.knownCanonical.chromEnd,hg38.knownCanonical.transcript,hg38.knownCanonical.protein,hg38.knownGeneMrna.name,hg38.knownGeneMrna.seq,id,3utr_len
362233,ENST00000263100.8,chr19,-,58345182,58353492,ENST00000263100.8,ugcagccgcgggcccagggugcuguugguguccucagaagugccgg...,A1BG,NM_130786,chr19,58345182.0,58353492.0,ENST00000263100.8,ENSG00000121410.12,ENST00000263100.8,ATTGCTGCAGACGCTCACCCCAGACACTCACTGCACCGGAGTGAGC...,A1BG|ENSG00000121410.12|ENST00000263100.8,1839
217289,ENST00000373997.8,chr10,-,50799408,50885627,ENST00000373997.8,agaugcuuuuuuaaauuuaagaauaagacacacaaaacucuauuaa...,A1CF,NM_014576,chr10,50799408.0,50885627.0,ENST00000373997.8,ENSG00000148584.16,ENST00000373997.8,ATAATCAAGGAAACCTTTTCCGGGTGGGGATCTCTGAAATTACTCA...,A1CF|ENSG00000148584.16|ENST00000373997.8,7320
249300,ENST00000318602.12,chr12,-,9067707,9115919,ENST00000318602.12,agaccacaaggcugaaaagugcuuugcuggaguccuguucucagag...,A2M,NM_001347425,chr12,9067707.0,9115919.0,ENST00000318602.12,ENSG00000175899.15,ENST00000318602.12,GGGACCAGATGGATTGTAGGGAGTAGGGTACAATACAGTCTGTTCT...,A2M|ENSG00000175899.15|ENST00000318602.12,115
249241,ENST00000299698.12,chr12,+,8822620,8876787,ENST00000299698.12,ggauaggagcuggaaacucaauuaguccucugugacauuuacugga...,A2ML1,NM_144670,chr12,8822620.0,8876787.0,ENST00000299698.12,ENSG00000166535.20,ENST00000299698.12,GACCCTGGAAAAATCTGTCTCACCCACAAAGATGTGGGCTCAGCTC...,A2ML1|ENSG00000166535.20|ENST00000299698.12,731
384208,ENST00000642412.2,chr22,-,42692120,42720870,ENST00000642412.2,ggggcccgccaggucaccuccccaaccugcuccugauggggcacug...,A4GALT,NM_017436,chr22,42692120.0,42720870.0,ENST00000642412.2,ENSG00000128274.17,ENST00000642412.2,ACTGCCCTCCGGCCGCCGCGCCGCCCGCCCGCCGGGGCCCCGCTGT...,A4GALT|ENSG00000128274.17|ENST00000642412.2,769
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10748,ENST00000371528.2,chr1,+,52842759,52894998,ENST00000371528.2,accuaaggaauuucagaggugugugcucuuccucaaugucaggugu...,ZYG11A,NM_001004339,chr1,52842759.0,52894998.0,ENST00000371528.2,ENSG00000203995.11,ENST00000371528.2,GCTAGCTCCGTGTGCCTCGCAGGCGTGGTGGGCGCGTCCTGGCAGC...,ZYG11A|ENSG00000203995.11|ENST00000371528.2,2041
10742,ENST00000294353.7,chr1,+,52726452,52827336,ENST00000294353.7,uagccauaaguauuggauaguugaaucacaggaauccuuuuuguga...,ZYG11B,NM_024646,chr1,52726452.0,52827336.0,ENST00000294353.7,ENSG00000162378.13,ENST00000294353.7,GGAGTCTGCGCTCTGGTTCGGGCTGCGGCTGCGGCTGCGGCTGCGG...,ZYG11B|ENSG00000162378.13|ENST00000294353.7,5707
162759,ENST00000322764.10,chr7,+,143381344,143391111,ENST00000322764.10,gugaggacaggcccucuucagaccgcaguccaugccccauugugga...,ZYX,NM_003461,chr7,143381344.0,143391111.0,ENST00000322764.10,ENSG00000159840.16,ENST00000322764.10,GCAGAGTCTGCGGACCCGGCGCCGAGGCGGCCACCCGAGACGCGGC...,ZYX|ENSG00000159840.16|ENST00000322764.10,429
318347,ENST00000381638.7,chr17,-,4004444,4143030,ENST00000381638.7,cccuugguggggcgugcaugggacuaguucaucugccacagggauu...,ZZEF1,NM_015113,chr17,4004444.0,4143030.0,ENST00000381638.7,ENSG00000074755.15,ENST00000381638.7,AGGAAGCCGGAAGCCGCAGGGGCCGCCGTCGTCTCCTCCGCGTCCC...,ZZEF1|ENSG00000074755.15|ENST00000381638.7,2445


In [21]:
# Proceed with previous pipeline

In [22]:
df = df.rename(columns={'hg38.knownGeneMrna.seq':'mrna'})
df

,"#""hg38.knownGene.name""",hg38.knownGene.chrom,hg38.knownGene.strand,hg38.knownGene.txStart,hg38.knownGene.txEnd,hg38.foldUtr3.name,hg38.foldUtr3.seq,hg38.kgXref.geneSymbol,hg38.kgXref.refseq,hg38.knownCanonical.chrom,hg38.knownCanonical.chromStart,hg38.knownCanonical.chromEnd,hg38.knownCanonical.transcript,hg38.knownCanonical.protein,hg38.knownGeneMrna.name,mrna,id,3utr_len
362233,ENST00000263100.8,chr19,-,58345182,58353492,ENST00000263100.8,ugcagccgcgggcccagggugcuguugguguccucagaagugccgg...,A1BG,NM_130786,chr19,58345182.0,58353492.0,ENST00000263100.8,ENSG00000121410.12,ENST00000263100.8,ATTGCTGCAGACGCTCACCCCAGACACTCACTGCACCGGAGTGAGC...,A1BG|ENSG00000121410.12|ENST00000263100.8,1839
217289,ENST00000373997.8,chr10,-,50799408,50885627,ENST00000373997.8,agaugcuuuuuuaaauuuaagaauaagacacacaaaacucuauuaa...,A1CF,NM_014576,chr10,50799408.0,50885627.0,ENST00000373997.8,ENSG00000148584.16,ENST00000373997.8,ATAATCAAGGAAACCTTTTCCGGGTGGGGATCTCTGAAATTACTCA...,A1CF|ENSG00000148584.16|ENST00000373997.8,7320
249300,ENST00000318602.12,chr12,-,9067707,9115919,ENST00000318602.12,agaccacaaggcugaaaagugcuuugcuggaguccuguucucagag...,A2M,NM_001347425,chr12,9067707.0,9115919.0,ENST00000318602.12,ENSG00000175899.15,ENST00000318602.12,GGGACCAGATGGATTGTAGGGAGTAGGGTACAATACAGTCTGTTCT...,A2M|ENSG00000175899.15|ENST00000318602.12,115
249241,ENST00000299698.12,chr12,+,8822620,8876787,ENST00000299698.12,ggauaggagcuggaaacucaauuaguccucugugacauuuacugga...,A2ML1,NM_144670,chr12,8822620.0,8876787.0,ENST00000299698.12,ENSG00000166535.20,ENST00000299698.12,GACCCTGGAAAAATCTGTCTCACCCACAAAGATGTGGGCTCAGCTC...,A2ML1|ENSG00000166535.20|ENST00000299698.12,731
384208,ENST00000642412.2,chr22,-,42692120,42720870,ENST00000642412.2,ggggcccgccaggucaccuccccaaccugcuccugauggggcacug...,A4GALT,NM_017436,chr22,42692120.0,42720870.0,ENST00000642412.2,ENSG00000128274.17,ENST00000642412.2,ACTGCCCTCCGGCCGCCGCGCCGCCCGCCCGCCGGGGCCCCGCTGT...,A4GALT|ENSG00000128274.17|ENST00000642412.2,769
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10748,ENST00000371528.2,chr1,+,52842759,52894998,ENST00000371528.2,accuaaggaauuucagaggugugugcucuuccucaaugucaggugu...,ZYG11A,NM_001004339,chr1,52842759.0,52894998.0,ENST00000371528.2,ENSG00000203995.11,ENST00000371528.2,GCTAGCTCCGTGTGCCTCGCAGGCGTGGTGGGCGCGTCCTGGCAGC...,ZYG11A|ENSG00000203995.11|ENST00000371528.2,2041
10742,ENST00000294353.7,chr1,+,52726452,52827336,ENST00000294353.7,uagccauaaguauuggauaguugaaucacaggaauccuuuuuguga...,ZYG11B,NM_024646,chr1,52726452.0,52827336.0,ENST00000294353.7,ENSG00000162378.13,ENST00000294353.7,GGAGTCTGCGCTCTGGTTCGGGCTGCGGCTGCGGCTGCGGCTGCGG...,ZYG11B|ENSG00000162378.13|ENST00000294353.7,5707
162759,ENST00000322764.10,chr7,+,143381344,143391111,ENST00000322764.10,gugaggacaggcccucuucagaccgcaguccaugccccauugugga...,ZYX,NM_003461,chr7,143381344.0,143391111.0,ENST00000322764.10,ENSG00000159840.16,ENST00000322764.10,GCAGAGTCTGCGGACCCGGCGCCGAGGCGGCCACCCGAGACGCGGC...,ZYX|ENSG00000159840.16|ENST00000322764.10,429
318347,ENST00000381638.7,chr17,-,4004444,4143030,ENST00000381638.7,cccuugguggggcgugcaugggacuaguucaucugccacagggauu...,ZZEF1,NM_015113,chr17,4004444.0,4143030.0,ENST00000381638.7,ENSG00000074755.15,ENST00000381638.7,AGGAAGCCGGAAGCCGCAGGGGCCGCCGTCGTCTCCTCCGCGTCCC...,ZZEF1|ENSG00000074755.15|ENST00000381638.7,2445


In [23]:
df['utr3'] = df.apply(lambda row: row['hg38.foldUtr3.seq'].upper().replace('U','T'), axis=1)
df

,"#""hg38.knownGene.name""",hg38.knownGene.chrom,hg38.knownGene.strand,hg38.knownGene.txStart,hg38.knownGene.txEnd,hg38.foldUtr3.name,hg38.foldUtr3.seq,hg38.kgXref.geneSymbol,hg38.kgXref.refseq,hg38.knownCanonical.chrom,hg38.knownCanonical.chromStart,hg38.knownCanonical.chromEnd,hg38.knownCanonical.transcript,hg38.knownCanonical.protein,hg38.knownGeneMrna.name,mrna,id,3utr_len,utr3
362233,ENST00000263100.8,chr19,-,58345182,58353492,ENST00000263100.8,ugcagccgcgggcccagggugcuguugguguccucagaagugccgg...,A1BG,NM_130786,chr19,58345182.0,58353492.0,ENST00000263100.8,ENSG00000121410.12,ENST00000263100.8,ATTGCTGCAGACGCTCACCCCAGACACTCACTGCACCGGAGTGAGC...,A1BG|ENSG00000121410.12|ENST00000263100.8,1839,TGCAGCCGCGGGCCCAGGGTGCTGTTGGTGTCCTCAGAAGTGCCGG...
217289,ENST00000373997.8,chr10,-,50799408,50885627,ENST00000373997.8,agaugcuuuuuuaaauuuaagaauaagacacacaaaacucuauuaa...,A1CF,NM_014576,chr10,50799408.0,50885627.0,ENST00000373997.8,ENSG00000148584.16,ENST00000373997.8,ATAATCAAGGAAACCTTTTCCGGGTGGGGATCTCTGAAATTACTCA...,A1CF|ENSG00000148584.16|ENST00000373997.8,7320,AGATGCTTTTTTAAATTTAAGAATAAGACACACAAAACTCTATTAA...
249300,ENST00000318602.12,chr12,-,9067707,9115919,ENST00000318602.12,agaccacaaggcugaaaagugcuuugcuggaguccuguucucagag...,A2M,NM_001347425,chr12,9067707.0,9115919.0,ENST00000318602.12,ENSG00000175899.15,ENST00000318602.12,GGGACCAGATGGATTGTAGGGAGTAGGGTACAATACAGTCTGTTCT...,A2M|ENSG00000175899.15|ENST00000318602.12,115,AGACCACAAGGCTGAAAAGTGCTTTGCTGGAGTCCTGTTCTCAGAG...
249241,ENST00000299698.12,chr12,+,8822620,8876787,ENST00000299698.12,ggauaggagcuggaaacucaauuaguccucugugacauuuacugga...,A2ML1,NM_144670,chr12,8822620.0,8876787.0,ENST00000299698.12,ENSG00000166535.20,ENST00000299698.12,GACCCTGGAAAAATCTGTCTCACCCACAAAGATGTGGGCTCAGCTC...,A2ML1|ENSG00000166535.20|ENST00000299698.12,731,GGATAGGAGCTGGAAACTCAATTAGTCCTCTGTGACATTTACTGGA...
384208,ENST00000642412.2,chr22,-,42692120,42720870,ENST00000642412.2,ggggcccgccaggucaccuccccaaccugcuccugauggggcacug...,A4GALT,NM_017436,chr22,42692120.0,42720870.0,ENST00000642412.2,ENSG00000128274.17,ENST00000642412.2,ACTGCCCTCCGGCCGCCGCGCCGCCCGCCCGCCGGGGCCCCGCTGT...,A4GALT|ENSG00000128274.17|ENST00000642412.2,769,GGGGCCCGCCAGGTCACCTCCCCAACCTGCTCCTGATGGGGCACTG...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10748,ENST00000371528.2,chr1,+,52842759,52894998,ENST00000371528.2,accuaaggaauuucagaggugugugcucuuccucaaugucaggugu...,ZYG11A,NM_001004339,chr1,52842759.0,52894998.0,ENST00000371528.2,ENSG00000203995.11,ENST00000371528.2,GCTAGCTCCGTGTGCCTCGCAGGCGTGGTGGGCGCGTCCTGGCAGC...,ZYG11A|ENSG00000203995.11|ENST00000371528.2,2041,ACCTAAGGAATTTCAGAGGTGTGTGCTCTTCCTCAATGTCAGGTGT...
10742,ENST00000294353.7,chr1,+,52726452,52827336,ENST00000294353.7,uagccauaaguauuggauaguugaaucacaggaauccuuuuuguga...,ZYG11B,NM_024646,chr1,52726452.0,52827336.0,ENST00000294353.7,ENSG00000162378.13,ENST00000294353.7,GGAGTCTGCGCTCTGGTTCGGGCTGCGGCTGCGGCTGCGGCTGCGG...,ZYG11B|ENSG00000162378.13|ENST00000294353.7,5707,TAGCCATAAGTATTGGATAGTTGAATCACAGGAATCCTTTTTGTGA...
162759,ENST00000322764.10,chr7,+,143381344,143391111,ENST00000322764.10,gugaggacaggcccucuucagaccgcaguccaugccccauugugga...,ZYX,NM_003461,chr7,143381344.0,143391111.0,ENST00000322764.10,ENSG00000159840.16,ENST00000322764.10,GCAGAGTCTGCGGACCCGGCGCCGAGGCGGCCACCCGAGACGCGGC...,ZYX|ENSG00000159840.16|ENST00000322764.10,429,GTGAGGACAGGCCCTCTTCAGACCGCAGTCCATGCCCCATTGTGGA...
318347,ENST00000381638.7,chr17,-,4004444,4143030,ENST00000381638.7,cccuugguggggcgugcaugggacuaguucaucugccacagggauu...,ZZEF1,NM_015113,chr17,4004444.0,4143030.0,ENST00000381638.7,ENSG00000074755.15,ENST00000381638.7,AGGAAGCCGGAAGCCGCAGGGGCCGCCGTCGTCTCCTCCGCGTCCC...,ZZEF1|ENSG00000074755.15|ENST00000381638.7,2445,CCCTTGGTGGGGCGTGCATGGGACTAGTTCATCTGCCACAGGGATT...


In [24]:
df_mrna = df[ ['id', 'mrna'] ]
df_mrna

,id,mrna
362233,A1BG|ENSG00000121410.12|ENST00000263100.8,ATTGCTGCAGACGCTCACCCCAGACACTCACTGCACCGGAGTGAGC...
217289,A1CF|ENSG00000148584.16|ENST00000373997.8,ATAATCAAGGAAACCTTTTCCGGGTGGGGATCTCTGAAATTACTCA...
249300,A2M|ENSG00000175899.15|ENST00000318602.12,GGGACCAGATGGATTGTAGGGAGTAGGGTACAATACAGTCTGTTCT...
249241,A2ML1|ENSG00000166535.20|ENST00000299698.12,GACCCTGGAAAAATCTGTCTCACCCACAAAGATGTGGGCTCAGCTC...
384208,A4GALT|ENSG00000128274.17|ENST00000642412.2,ACTGCCCTCCGGCCGCCGCGCCGCCCGCCCGCCGGGGCCCCGCTGT...
...,...,...
10748,ZYG11A|ENSG00000203995.11|ENST00000371528.2,GCTAGCTCCGTGTGCCTCGCAGGCGTGGTGGGCGCGTCCTGGCAGC...
10742,ZYG11B|ENSG00000162378.13|ENST00000294353.7,GGAGTCTGCGCTCTGGTTCGGGCTGCGGCTGCGGCTGCGGCTGCGG...
162759,ZYX|ENSG00000159840.16|ENST00000322764.10,GCAGAGTCTGCGGACCCGGCGCCGAGGCGGCCACCCGAGACGCGGC...
318347,ZZEF1|ENSG00000074755.15|ENST00000381638.7,AGGAAGCCGGAAGCCGCAGGGGCCGCCGTCGTCTCCTCCGCGTCCC...


In [25]:
len(df_mrna['mrna'])-len(df_mrna['mrna'].drop_duplicates())

0

In [26]:
print(df_mrna[df_mrna.duplicated(['mrna'], keep=False)]['id'].tolist())

[]


In [27]:
df_utr3 = df[ ['id', 'utr3'] ]
df_utr3

,id,utr3
362233,A1BG|ENSG00000121410.12|ENST00000263100.8,TGCAGCCGCGGGCCCAGGGTGCTGTTGGTGTCCTCAGAAGTGCCGG...
217289,A1CF|ENSG00000148584.16|ENST00000373997.8,AGATGCTTTTTTAAATTTAAGAATAAGACACACAAAACTCTATTAA...
249300,A2M|ENSG00000175899.15|ENST00000318602.12,AGACCACAAGGCTGAAAAGTGCTTTGCTGGAGTCCTGTTCTCAGAG...
249241,A2ML1|ENSG00000166535.20|ENST00000299698.12,GGATAGGAGCTGGAAACTCAATTAGTCCTCTGTGACATTTACTGGA...
384208,A4GALT|ENSG00000128274.17|ENST00000642412.2,GGGGCCCGCCAGGTCACCTCCCCAACCTGCTCCTGATGGGGCACTG...
...,...,...
10748,ZYG11A|ENSG00000203995.11|ENST00000371528.2,ACCTAAGGAATTTCAGAGGTGTGTGCTCTTCCTCAATGTCAGGTGT...
10742,ZYG11B|ENSG00000162378.13|ENST00000294353.7,TAGCCATAAGTATTGGATAGTTGAATCACAGGAATCCTTTTTGTGA...
162759,ZYX|ENSG00000159840.16|ENST00000322764.10,GTGAGGACAGGCCCTCTTCAGACCGCAGTCCATGCCCCATTGTGGA...
318347,ZZEF1|ENSG00000074755.15|ENST00000381638.7,CCCTTGGTGGGGCGTGCATGGGACTAGTTCATCTGCCACAGGGATT...


In [28]:
df_mrna.to_csv('/home/kwonschul/data/genome/salmon_hg38_canonical_250114/Canonical_hg38_260630_mrna.txt', sep='\t', index=False)
df_utr3.to_csv('/home/kwonschul/data/genome/salmon_hg38_canonical_250114/Canonical_hg38_260630_utr3.txt', sep='\t', index=False)


In [29]:
# linux
# seqkit tab2fx -p id Canonical_hg38_240723_mrna.txt > Canonical_hg38_240723_mrna.fa
